# Add Text Features to the Training Dataset

Turns the stock-day sentence embeddings produced by
`01 - feature extraction/features_08_text_embeddings.ipynb` into model inputs and attaches
them to `merged_master.pkl`. The prediction pipeline itself is untouched: this notebook writes
a **separate** file, `merged_master_text=<mode>.pkl`, that any `*_all_features` model notebook
can be pointed at by setting `TEXT_VARIANT = "<mode>"`. The 04/05 notebooks resolve the
matching `predictions_*_text=<mode>.pkl` files the same way.

**Three ways to represent the text (`TEXT_MODE`)**

| mode | columns added | what it is | uses returns? |
|---|---|---|---|
| `raw` | `text_emb_000` .. `text_emb_383`, `text_n` | the 384 mean-pooled dimensions, unchanged | no |
| `pca` | `text_pc_1` .. `text_pc_K`, `text_n` | projections on the top-K principal directions of the stock-day vectors, loadings fit on the pre-OOS period and applied forward | no |
| `supervised` | `text_score`, `text_n` | walk-forward ridge prediction of the (cross-sectionally de-meaned) next-day target from the 384 dimensions; the score for month *m* is fit on the `SUP_WINDOW` trading days before *m* only | yes, walk-forward |

`text_n` is the number of message-symbol pairs behind the stock-day (0 = no messages).
Stock-days without messages get 0 in every text column: a zero vector in `raw`, the fit-sample
mean in `pca` (components are centred), and "no text signal" in `supervised`. Models can
separate the two cases through `text_n`.

**Why the 384 raw dimensions are not simply merged into `merged_master.pkl`:** 16.8M rows x
384 float32 is ~26 GB before any model notebook copies it. `raw` mode is implemented and
works in principle, but on a 64 GB machine the all-features regressions cannot hold that
frame; `pca` and `supervised` are the practical routes, with `raw` available for subsamples
or a larger machine.

## 1. Setup and Configuration

In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# =============================================================================
# DIRECTORY PATHS
# =============================================================================
DATA_DIR       = Path(r"C:\Users\skazempour\Documents\StockTwits\Data\v1\data\csv")
MODEL_DATA_DIR = Path(r"C:\Users\skazempour\Documents\StockTwits\Data")
EMBED_FILE     = DATA_DIR / "text_embeddings_mlcrowd" / "text_embeddings_stock_day.pkl"
INPUT_DATA     = MODEL_DATA_DIR / "merged_master.pkl"

# =============================================================================
# PARAMETERS
# =============================================================================
TEXT_MODE = "supervised"        # "raw" | "pca" | "supervised"

# --- pca -----------------------------------------------------------------------
N_COMPONENTS = 5
PCA_FIT_END  = "2011-12-31"     # loadings are fit on stock-days up to here (the pre-OOS period) and applied forward

# --- supervised ----------------------------------------------------------------
SUP_TARGET         = "f_cumret1"   # the prediction notebooks' next-day target; e.g. "ar_capm_1" for CAPM-abnormal
SUP_TRAIN_END_DATE = "2010-12-31"  # first scored month is the following one. One year before 03a's own
                                   # TRAIN_END_DATE so that scores already exist inside 03a's first rolling window.
SUP_WINDOW         = 252           # trading days per rolling fit, matching the prediction notebooks
RIDGE_ALPHA        = 100.0         # L2 penalty on standardised embedding columns
MIN_TRAIN_ROWS     = 10 * 384      # skip a month if fewer training rows than this are available

OUTPUT_FILE = MODEL_DATA_DIR / f"merged_master_text={TEXT_MODE}.pkl"
META_FILE   = MODEL_DATA_DIR / f"merged_master_text={TEXT_MODE}.json"

assert TEXT_MODE in ("raw", "pca", "supervised"), TEXT_MODE
print(f"Mode        : {TEXT_MODE}")
print(f"Embeddings  : {EMBED_FILE}")
print(f"Input       : {INPUT_DATA}")
print(f"Output      : {OUTPUT_FILE}")

## 2. Load the Stock-Day Embeddings

In [ ]:
emb = pd.read_pickle(EMBED_FILE)
EMBED_COLS = [c for c in emb.columns if c.startswith("embed_") and c != "embed_n"]
EMBED_DIM = len(EMBED_COLS)
emb["date"] = pd.to_datetime(emb["date"])
emb["symbol"] = emb["symbol"].astype(object)
assert not emb.duplicated(["symbol", "date"]).any()

X_all = emb[EMBED_COLS].to_numpy(dtype=np.float32)
emb_dates = emb["date"].to_numpy()
print(f"Stock-day embeddings: {len(emb):,} rows x {EMBED_DIM} dims, "
      f"{emb['date'].min().date()} to {emb['date'].max().date()}, {emb['symbol'].nunique():,} symbols")
print(f"Messages per stock-day: median {emb['embed_n'].median():.0f}, max {emb['embed_n'].max():,}")

## 3. Load the Training Dataset and Align the Keys

`merged_master.pkl` is keyed by (`ticker`, `date`) on the CRSP side; the embeddings by
(`symbol`, `date`). `perpare_training_data.ipynb` matches the two on `ticker == symbol`, and
so does this cell, via a compact integer key rather than a `pd.merge` on the 16.8M-row frame.
`pos[i]` is the embedding row for training row `i`, or -1 when that stock-day has no messages.

In [ ]:
t0 = time.time()
mm = pd.read_pickle(INPUT_DATA)
mm["date"] = pd.to_datetime(mm["date"])
print(f"Training dataset: {len(mm):,} rows x {mm.shape[1]} columns (loaded in {time.time() - t0:.0f}s)")

sym_all = pd.Index(pd.unique(np.concatenate([emb["symbol"].to_numpy(), mm["ticker"].astype(object).to_numpy()])))

def make_key(symbols, dates):
    s = sym_all.get_indexer(symbols).astype(np.int64)
    d = (pd.DatetimeIndex(dates).asi8 // 86_400_000_000_000).astype(np.int64)
    return s * (1 << 32) + d

emb_key = pd.Index(make_key(emb["symbol"].to_numpy(), emb["date"].to_numpy()))
pos = emb_key.get_indexer(make_key(mm["ticker"].astype(object).to_numpy(), mm["date"].to_numpy()))
has_text = pos >= 0

print(f"Training rows with text: {has_text.sum():,} of {len(mm):,} ({has_text.mean():.1%})")
print(f"Embedding rows matched to at least one training row: "
      f"{len(np.unique(pos[has_text])):,} of {len(emb):,} ({len(np.unique(pos[has_text])) / len(emb):.1%})")

## 4. Build the Text Features (one block per mode)

Each builder returns a DataFrame with one row per **embedding** row (same order as `emb`).
Section 5 scatters it onto the training rows through `pos`.

In [ ]:
def build_raw():
    cols = [f"text_emb_{i:03d}" for i in range(EMBED_DIM)]
    return pd.DataFrame(X_all, columns=cols), {"columns": cols}


def build_pca():
    fit_mask = emb_dates <= np.datetime64(pd.Timestamp(PCA_FIT_END))
    Xfit = X_all[fit_mask].astype(np.float64)
    assert len(Xfit) > EMBED_DIM, f"only {len(Xfit)} stock-days up to {PCA_FIT_END}; extend PCA_FIT_END"
    mu = Xfit.mean(axis=0)
    Xc = Xfit - mu
    cov = Xc.T @ Xc / (len(Xc) - 1)
    evals, evecs = np.linalg.eigh(cov)                 # ascending
    order = np.argsort(evals)[::-1]
    evals, evecs = evals[order], evecs[:, order]
    comps = evecs[:, :N_COMPONENTS]                    # (EMBED_DIM, K)
    # Sign convention: make the largest-magnitude loading of each component positive
    comps = comps * np.sign(comps[np.abs(comps).argmax(axis=0), np.arange(N_COMPONENTS)])
    explained = evals[:N_COMPONENTS] / evals.sum()
    pcs = ((X_all.astype(np.float64) - mu) @ comps).astype(np.float32)
    cols = [f"text_pc_{k + 1}" for k in range(N_COMPONENTS)]
    print(f"PCA fit on {len(Xfit):,} stock-days up to {PCA_FIT_END}")
    print("Explained variance per component: " + ", ".join(f"{e:.1%}" for e in explained)
          + f"  (total {explained.sum():.1%})")
    info = {"columns": cols, "fit_end": PCA_FIT_END, "n_fit": int(len(Xfit)),
            "explained_variance_ratio": [float(e) for e in explained]}
    return pd.DataFrame(pcs, columns=cols), info


def build_supervised():
    # --- target per embedding row: the training row it matches (first one if a ticker maps to
    #     more than one permno on a date), cross-sectionally de-meaned by date ---
    hit = np.flatnonzero(has_text)
    first_hit = pd.Series(hit).groupby(pos[hit]).first()
    y = np.full(len(emb), np.nan)
    y[first_hit.index.to_numpy()] = mm[SUP_TARGET].to_numpy(dtype=float, na_value=np.nan)[first_hit.to_numpy()]
    y_dm = y - pd.Series(y).groupby(emb_dates).transform("mean").to_numpy()
    print(f"Embedding rows with a {SUP_TARGET} label: {np.isfinite(y_dm).sum():,} of {len(emb):,}")

    # --- walk-forward ridge, monthly refit, same window/cutoff conventions as the 03a notebooks ---
    dates = pd.DatetimeIndex(np.sort(np.unique(emb_dates)))
    train_end = pd.Timestamp(SUP_TRAIN_END_DATE)
    months = sorted(dates[dates > train_end].to_period("M").unique())
    month_arr = emb["date"].dt.to_period("M").to_numpy()
    score = np.full(len(emb), np.nan, dtype=np.float32)
    eye = np.eye(EMBED_DIM)
    fit_log = []

    for m in tqdm(months, desc="Walk-forward ridge"):
        first_day = m.to_timestamp()
        train_cutoff = first_day - pd.Timedelta(days=2)
        tr_dates = dates[dates <= train_cutoff]
        if len(tr_dates) < SUP_WINDOW:
            continue
        start, last = tr_dates[-SUP_WINDOW], tr_dates[-1]
        tr = (emb_dates >= np.datetime64(start)) & (emb_dates <= np.datetime64(last)) & np.isfinite(y_dm)
        te = month_arr == m
        if tr.sum() < MIN_TRAIN_ROWS or te.sum() == 0:
            continue

        Xtr = X_all[tr].astype(np.float64)
        ytr = y_dm[tr]
        mu, sd = Xtr.mean(axis=0), Xtr.std(axis=0) + 1e-8
        Z = (Xtr - mu) / sd
        ybar = ytr.mean()
        beta = np.linalg.solve(Z.T @ Z + RIDGE_ALPHA * eye, Z.T @ (ytr - ybar))
        Zte = (X_all[te].astype(np.float64) - mu) / sd
        score[te] = (Zte @ beta + ybar).astype(np.float32)

        resid = ytr - ybar - Z @ beta
        fit_log.append({"month": str(m), "train_start": start.date(), "train_end": last.date(),
                        "n_train": int(tr.sum()), "n_scored": int(te.sum()),
                        "r2_in_sample": float(1 - (resid ** 2).sum() / ((ytr - ybar) ** 2).sum())})

    fit_df = pd.DataFrame(fit_log)
    print(f"Scored months: {len(fit_df)} ({fit_df['month'].iloc[0]} to {fit_df['month'].iloc[-1]}); "
          f"scored stock-days: {np.isfinite(score).sum():,}; "
          f"median in-sample R2 {fit_df['r2_in_sample'].median():.4f}")
    display(fit_df.head())
    info = {"columns": ["text_score"], "target": SUP_TARGET, "train_end_date": SUP_TRAIN_END_DATE,
            "window": SUP_WINDOW, "ridge_alpha": RIDGE_ALPHA, "n_months": int(len(fit_df)),
            "median_r2_in_sample": float(fit_df["r2_in_sample"].median())}
    return pd.DataFrame({"text_score": score}), info


builders = {"raw": build_raw, "pca": build_pca, "supervised": build_supervised}
t0 = time.time()
text_feats, info = builders[TEXT_MODE]()
TEXT_COLS = info["columns"]
assert len(text_feats) == len(emb)
print(f"\nBuilt {len(TEXT_COLS)} text column(s) in {time.time() - t0:.0f}s: "
      f"{TEXT_COLS[:5]}{' ...' if len(TEXT_COLS) > 5 else ''}")

## 5. Attach to the Training Rows and Save

Columns are added one at a time (no copy of the existing frame). Rows without messages get 0
and `text_n = 0`.

In [ ]:
if TEXT_MODE == "raw":
    print(f"WARNING: raw mode adds ~{len(mm) * EMBED_DIM * 4 / 1024**3:.0f} GB of float32 columns to the "
          f"{len(mm):,}-row frame. Make sure the machine can hold it (and that the model notebooks can).")

idx = pos[has_text]
feat_vals = text_feats.to_numpy(dtype=np.float32)
for j, c in enumerate(tqdm(TEXT_COLS, desc="Attaching text columns")):
    col = np.zeros(len(mm), dtype=np.float32)
    col[has_text] = np.nan_to_num(feat_vals[idx, j], nan=0.0)   # no value in this mode -> 0, like no messages
    mm[c] = col
text_n = np.zeros(len(mm), dtype=np.int32)
text_n[has_text] = emb["embed_n"].to_numpy()[idx]
mm["text_n"] = text_n
del feat_vals

nan_in_text = int(np.isnan(text_feats.to_numpy(dtype=np.float32)).any(axis=1).sum())
print(f"\nOutput: {len(mm):,} rows x {mm.shape[1]} columns")
print(f"Text columns: {TEXT_COLS + ['text_n']}"[:300])
print(f"Rows with text_n > 0: {(mm['text_n'] > 0).sum():,} ({(mm['text_n'] > 0).mean():.1%})")
if nan_in_text:
    print(f"Note: {nan_in_text:,} embedding rows had no value in this mode (e.g. pre-OOS months in "
          f"'supervised'); they were written as 0.")
print(mm[TEXT_COLS[:5] + ["text_n"]].describe().T.round(5))

print(f"\nSaving to {OUTPUT_FILE} ...")
mm.to_pickle(OUTPUT_FILE)
meta = {"mode": TEXT_MODE, "embeddings": str(EMBED_FILE), "input": str(INPUT_DATA),
        "n_rows": int(len(mm)), "share_rows_with_text": float(has_text.mean()),
        "text_columns": TEXT_COLS + ["text_n"], **{k: v for k, v in info.items() if k != "columns"},
        "created": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}
META_FILE.write_text(json.dumps(meta, indent=2, default=str))
print(f"Saved ({OUTPUT_FILE.stat().st_size / 1024**3:.1f} GB); metadata in {META_FILE.name}")

## How the rest of the pipeline picks this up

1. In any `*_all_features` model notebook (03a, 03d, 03e) set `TEXT_VARIANT = "<mode>"`.
   It then reads `merged_master_text=<mode>.pkl`, includes the `text_*` columns as features
   (they are numeric and not on the exclusion list), and writes
   `predictions_<model>_input=<N>_text=<mode>.pkl` so the baseline prediction files are
   never overwritten.
2. In `04 - predictive regressions/*` and `05 - trading/form_portfolios.ipynb` set the same
   `TEXT_VARIANT`; `find_all_features_file()` then resolves the `_text=<mode>` prediction
   files (and ignores them when `TEXT_VARIANT = None`).
3. To compare representations, run this notebook once per mode, run the model notebook once
   per `TEXT_VARIANT`, and switch `TEXT_VARIANT` in the 04/05 notebooks.

**Look-ahead discipline.** `raw` and `pca` never see a return. `pca`'s loadings are fit on
the pre-OOS period only. `supervised` refits monthly on a trailing window that ends before
the scored month, mirroring the model notebooks; a score dated *t* uses returns realised no
later than the prediction notebooks themselves would.